In [0]:
from pyspark.sql.types import StructType, StructField,StringType ,IntegerType, DateType, TimestampType, FloatType,DecimalType
import pyspark.sql.functions as F

In [0]:
catalog_name = "ecommerce"

## Brands


In [0]:
df_bronze =spark.table(f"{catalog_name}.bronze.brz_brands")


brand_code,brand_name,category_code,source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
VOLT@,VoltEdge,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z


In [0]:
df_bronze = df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")))


brand_code,brand_name,category_code,source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
VOLT@,VoltEdge,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z


In [0]:
df_bronze = df_bronze.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r"[^A-Za-z0-9]", ""))


brand_code,brand_name,category_code,source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
VOLT,VoltEdge,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z


In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("brand_code", "brand_name")
df_duplicates = df_bronze.withColumn("dup_count", F.count("*").over(window_spec)) \
    .filter(F.col("dup_count") > 1)



brand_code,brand_name,category_code,source_file,ingested_at,dup_count


In [0]:
anomalies = {
    "GROCERY" : "GRCY",
    "BOOKS" : "BKS",
    "TOYS": "TOY"
}
df_bronze = df_bronze.replace(to_replace=anomalies,subset=["category_code"])


brand_code,brand_name,category_code,source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
VOLT,VoltEdge,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-02-16T03:56:26.304Z


In [0]:
df_silver = df_bronze

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

## Category

In [0]:
df_bronze_cat = spark.table(f"{catalog_name}.bronze.brz_category")


category_code,category_name,source_file,ingested_at
ce,Electronics,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
app,Apparel,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
bks,Books,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
grcy,Grocery,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
toy,Toys & Games,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
spt,Sports & Outdoors,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
app,Apparel,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
grcy,Grocery,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z


In [0]:
window_spec = Window.partitionBy("category_code","category_name")
df_duplicates_cat = df_bronze_cat.withColumn("dup_count",F.count("*").over(window_spec)).filter(F.col("dup_count")>1)



category_code,category_name,source_file,ingested_at,dup_count
app,Apparel,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z,2
app,Apparel,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z,2
grcy,Grocery,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z,2
grcy,Grocery,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z,2


In [0]:
df_silver_cat = df_bronze_cat.dropDuplicates(['category_code'])


category_code,category_name,source_file,ingested_at
ce,Electronics,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
app,Apparel,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
bks,Books,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
grcy,Grocery,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
toy,Toys & Games,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
spt,Sports & Outdoors,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z


In [0]:
df_silver_cat = df_silver_cat.withColumn("category_code",F.upper(F.col("category_code")))


category_code,category_name,source_file,ingested_at
CE,Electronics,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
APP,Apparel,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
HNK,Home & Kitchen,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
BPC,Beauty & Personal Care,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
BKS,Books,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
GRCY,Grocery,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
TOY,Toys & Games,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z
SPT,Sports & Outdoors,dbfs:/Volumes/ecommerce/raw/raw_landing/category/category.csv,2026-02-16T04:14:15.858Z


In [0]:
df_silver_cat.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

## products

In [0]:
df_bronze_products = spark.table(f"{catalog_name}.bronze.brz_products")

                                

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000060,SLKE-BPC-00006,bpc,slke,Purple,One-Size,Plastic,232g,"28,0",13.8,6.1,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000077,VOLT-CE-00007,ce,volt,Blue,One-Size,Plastic,507g,"27,2",12.1,6.4,5,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000084,CBLT-APP-00008,app,cblt,Blue,XS,Polyester,261g,"27,7",8.5,7.0,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000091,ARFT-SPT-00009,spt,arft,Blue,XL,Plastic,59g,"12,5",19.0,7.9,11,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000107,MOSA-APP-0000A,app,mosa,White,L,Polyester,238g,"10,7",17.7,10.3,6,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z


In [0]:
df_bronze_products = df_bronze_products.withColumn("category_code",F.upper(F.col("category_code")))



product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,HNK,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000022,HMNS-HNK-00002,HNK,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000039,NOVW-CE-00003,CE,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000046,URTL-APP-00004,APP,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000053,GGRN-GRC-00005,GRCY,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000060,SLKE-BPC-00006,BPC,slke,Purple,One-Size,Plastic,232g,"28,0",13.8,6.1,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000077,VOLT-CE-00007,CE,volt,Blue,One-Size,Plastic,507g,"27,2",12.1,6.4,5,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000084,CBLT-APP-00008,APP,cblt,Blue,XS,Polyester,261g,"27,7",8.5,7.0,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000091,ARFT-SPT-00009,SPT,arft,Blue,XL,Plastic,59g,"12,5",19.0,7.9,11,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000107,MOSA-APP-0000A,APP,mosa,White,L,Polyester,238g,"10,7",17.7,10.3,6,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z


In [0]:
df_bronze_products = df_bronze_products.withColumn("brand_code",F.upper(F.col("brand_code")))


product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000060,SLKE-BPC-00006,BPC,SLKE,Purple,One-Size,Plastic,232g,"28,0",13.8,6.1,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000077,VOLT-CE-00007,CE,VOLT,Blue,One-Size,Plastic,507g,"27,2",12.1,6.4,5,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000084,CBLT-APP-00008,APP,CBLT,Blue,XS,Polyester,261g,"27,7",8.5,7.0,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000091,ARFT-SPT-00009,SPT,ARFT,Blue,XL,Plastic,59g,"12,5",19.0,7.9,11,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000107,MOSA-APP-0000A,APP,MOSA,White,L,Polyester,238g,"10,7",17.7,10.3,6,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z


In [0]:
df_bronze_products.createOrReplaceTempView("vw_bronze_products")

In [0]:
df_distinct_material = df_bronze_products.select("material").distinct()


material
Coton
Steel
Wood
Ruber
Plastic
Polyester
Glass
Alumium
Paper
Leather


In [0]:
df_bronze_products=df_bronze_products.replace(to_replace = "Ruber",value= "Rubber",subset=["material"])


product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Rubber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Rubber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000060,SLKE-BPC-00006,BPC,SLKE,Purple,One-Size,Plastic,232g,"28,0",13.8,6.1,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000077,VOLT-CE-00007,CE,VOLT,Blue,One-Size,Plastic,507g,"27,2",12.1,6.4,5,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000084,CBLT-APP-00008,APP,CBLT,Blue,XS,Polyester,261g,"27,7",8.5,7.0,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000091,ARFT-SPT-00009,SPT,ARFT,Blue,XL,Plastic,59g,"12,5",19.0,7.9,11,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000107,MOSA-APP-0000A,APP,MOSA,White,L,Polyester,238g,"10,7",17.7,10.3,6,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z


In [0]:
df_bronze_products = df_bronze_products.withColumn("length_cm", F.regexp_replace(F.col("length_cm"), ",", "."))



product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Coton,305g,22.2,17.1,6.3,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682g,18.2,12.3,3.7,1,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243g,18.2,13.9,4.2,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Rubber,225g,17.6,4.6,5.8,50,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Rubber,455g,27.2,15.8,7.4,-4,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000060,SLKE-BPC-00006,BPC,SLKE,Purple,One-Size,Plastic,232g,28.0,13.8,6.1,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000077,VOLT-CE-00007,CE,VOLT,Blue,One-Size,Plastic,507g,27.2,12.1,6.4,5,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000084,CBLT-APP-00008,APP,CBLT,Blue,XS,Polyester,261g,27.7,8.5,7.0,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000091,ARFT-SPT-00009,SPT,ARFT,Blue,XL,Plastic,59g,12.5,19.0,7.9,11,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000107,MOSA-APP-0000A,APP,MOSA,White,L,Polyester,238g,10.7,17.7,10.3,6,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z


In [0]:
df_bronze_products = df_bronze_products \
    .withColumn("height_cm", F.col("height_cm").cast(DecimalType(10,2))) \
    .withColumn("width_cm", F.col("length_cm").cast(DecimalType(10,2)))


product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,HNK,STCR,White,One-Size,Coton,305g,22.2,22.20,6.30,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000022,HMNS-HNK-00002,HNK,HMNS,Silver,One-Size,Steel,682g,18.2,18.20,3.70,1,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000039,NOVW-CE-00003,CE,NOVW,Purple,One-Size,Wood,243g,18.2,18.20,4.20,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000046,URTL-APP-00004,APP,URTL,Silver,S,Rubber,225g,17.6,17.60,5.80,50,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000053,GGRN-GRC-00005,GRCY,GGRN,Silver,One-Size,Rubber,455g,27.2,27.20,7.40,-4,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000060,SLKE-BPC-00006,BPC,SLKE,Purple,One-Size,Plastic,232g,28.0,28.00,6.10,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000077,VOLT-CE-00007,CE,VOLT,Blue,One-Size,Plastic,507g,27.2,27.20,6.40,5,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000084,CBLT-APP-00008,APP,CBLT,Blue,XS,Polyester,261g,27.7,27.70,7.00,0,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000091,ARFT-SPT-00009,SPT,ARFT,Blue,XL,Plastic,59g,12.5,12.50,7.90,11,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z
2000000000107,MOSA-APP-0000A,APP,MOSA,White,L,Polyester,238g,10.7,10.70,10.30,6,dbfs:/Volumes/ecommerce/raw/raw_landing/products/products.csv,2026-02-16T04:17:25.636Z


In [0]:
df_silver = df_bronze_products.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

+------------+
|weight_grams|
+------------+
|305         |
|682         |
|243         |
|225         |
|455         |
+------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)
df_silver.select("material").distinct().show() 

+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
| Aluminum|
|    Paper|
|  Leather|
+---------+



In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
# Check final cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

+------------+---------+-------------+----------+---------+------------+
|weight_grams|length_cm|category_code|brand_code|material |rating_count|
+------------+---------+-------------+----------+---------+------------+
|305         |22.2     |HNK          |STCR      |Cotton   |0           |
|682         |18.2     |HNK          |HMNS      |Steel    |1           |
|243         |18.2     |CE           |NOVW      |Wood     |0           |
|225         |17.6     |APP          |URTL      |Rubber   |50          |
|455         |27.2     |GRCY         |GGRN      |Rubber   |4           |
|232         |28.0     |BPC          |SLKE      |Plastic  |0           |
|507         |27.2     |CE           |VOLT      |Plastic  |5           |
|261         |27.7     |APP          |CBLT      |Polyester|0           |
|59          |12.5     |SPT          |ARFT      |Plastic  |11          |
|238         |10.7     |APP          |MOSA      |Polyester|6           |
+------------+---------+-------------+----------+--

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

## Customers

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")



Row count: 300000
Column count: 7
+----------------+--------------+------------+--------------+-----+--------------------+--------------------+
|     customer_id|         phone|country_code|       country|state|           file_name|    ingest_timestamp|
+----------------+--------------+------------+--------------+-----+--------------------+--------------------+
|CUST000000000001|917280033536.0|          IN|         India|   MH|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000002|619489725433.0|          AU|     Australia|  VIC|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000003|919390066524.0|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000004|917073741793.0|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000005|618478772532.0|          AU|     Australia|   WA|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000006|916441718520.0|          IN|         India|   GJ|dbfs:/Volumes/eco..

In [0]:
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

300

In [0]:
# There are 300 null values in customer_id column. Display some of those
df_bronze.filter(F.col("customer_id").isNull()).show(3)

+-----------+--------------+------------+-------+-----+--------------------+--------------------+
|customer_id|         phone|country_code|country|state|           file_name|    ingest_timestamp|
+-----------+--------------+------------+-------+-----+--------------------+--------------------+
|       NULL|918187043562.0|          IN|  India|   DL|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|       NULL|917517243052.0|          IN|  India|   DL|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|       NULL|          NULL|          IN|  India|   GJ|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
+-----------+--------------+------------+-------+-----+--------------------+--------------------+
only showing top 3 rows


In [0]:
# Drop rows where 'customer_id' is null
df_silver = df_bronze.dropna(subset=["customer_id"])

# Get row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

Row count after droping null values: 299700


In [0]:
null_count = df_silver.filter(F.col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}") 

Number of nulls in phone: 29964


In [0]:
df_silver.filter(F.col("phone").isNull()).show(3)

+----------------+-----+------------+-------+-----+--------------------+--------------------+
|     customer_id|phone|country_code|country|state|           file_name|    ingest_timestamp|
+----------------+-----+------------+-------+-----+--------------------+--------------------+
|CUST000000000007| NULL|          IN|  India|   MH|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000010| NULL|          IN|  India|   RJ|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
|CUST000000000026| NULL|          IN|  India|   WB|dbfs:/Volumes/eco...|2026-02-16 04:23:...|
+----------------+-----+------------+-------+-----+--------------------+--------------------+
only showing top 3 rows


In [0]:
### Fill null values with 'Not Available'
df_silver = df_silver.fillna("Not Available", subset=["phone"])

# sanity check (If any nulls still exist)
df_silver.filter(F.col("phone").isNull()).show()

+-----------+-----+------------+-------+-----+---------+----------------+
|customer_id|phone|country_code|country|state|file_name|ingest_timestamp|
+-----------+-----+------------+-------+-----+---------+----------------+
+-----------+-----+------------+-------+-----+---------+----------------+



In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_customers)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

## Calender/Date

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")



Row count: 741
Column count: 7
+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+---------+-------+------------+--------------------+--------------------+
|2024-01-01|2024|   monday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-02|2024|  tuesday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-03|2024|WEDNESDAY|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
+----------+----+---------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
# Find duplicate rows in the DataFrame
duplicates = df_bronze.groupBy('date').count().filter("count > 1")

# Show the duplicate rows
print("Total duplicated Rows: ", duplicates.count())


Total duplicated Rows:  10


date,count
2024-02-03,2
2024-02-09,2
2024-10-27,2
2024-12-22,2
2025-04-01,2
2025-05-29,2
2025-07-11,2
2025-07-13,2
2025-09-25,2
2025-12-04,2


In [0]:
# Remove duplicate rows
df_silver = df_bronze.dropDuplicates(['date'])

# Get row count
row_count = df_silver.count()

print("Rows After removing Duplicates: ", row_count)

Rows After removing Duplicates:  731


In [0]:
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))


+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+---------+-------+------------+--------------------+--------------------+
|2024-01-01|2024|   Monday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-02|2024|  Tuesday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-03|2024|Wednesday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-04|2024| Thursday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-05|2024|   Friday|      1|          -1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
+----------+----+---------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))  # Convert negative to positive


+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+---------+-------+------------+--------------------+--------------------+
|2024-01-01|2024|   Monday|      1|           1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-02|2024|  Tuesday|      1|           1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-03|2024|Wednesday|      1|           1|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
+----------+----+---------+-------+------------+--------------------+--------------------+
only showing top 3 rows


Enhance quarter and weak_of_year column


In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))


+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+---------+-------+------------+--------------------+--------------------+
|2024-01-01|2024|   Monday|Q1-2024|  Week1-2024|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-02|2024|  Tuesday|Q1-2024|  Week1-2024|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
|2024-01-03|2024|Wednesday|Q1-2024|  Week1-2024|2026-02-16 04:23:...|dbfs:/Volumes/eco...|
+----------+----+---------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
# Rename a column
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")